# Cascading Failures

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryouy/simulations/blob/main/simulations/cascading-failures/cascading-failures.ipynb)

故障が連鎖するネットワークの耐障害性モデル。 最初のコードセルを実行し、最後のセルで設定・実行します。

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image, clear_output
from tqdm.auto import tqdm
import ipywidgets as widgets
from pathlib import Path
import json, csv, math, time

MODE = "failure"
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
REPO = "ryouy/simulations"
PREFIX = "simulations/cascading-failures"

def run_model(n, steps, seed, graph_type, degree, p1, p2):
    rng = np.random.default_rng(seed)
    if graph_type == "Scale-free":
        graph = nx.barabasi_albert_graph(n, max(1,min(degree,n-1)), seed=seed)
    elif graph_type == "Small-world":
        k = min(n-1, max(2, 2*math.ceil(degree/2)))
        if k%2: k-=1
        graph = nx.watts_strogatz_graph(n,k,.12,seed=seed)
    else:
        graph = nx.erdos_renyi_graph(n,min(1,degree/(n-1)),seed=seed)
    positions = nx.spring_layout(graph,seed=seed,iterations=45)
    neighbors = [np.fromiter(graph.neighbors(i),dtype=int) for i in range(n)]
    if MODE == "diffusion":
        values = np.zeros(n,dtype=float)
        values[int(np.argmax([graph.degree(i) for i in range(n)]))] = 1
    elif MODE == "opinion":
        values = rng.choice([-1.,1.],n) + rng.normal(0,.12,n)
        values = np.clip(values,-1,1)
    else:
        values = np.zeros(n,dtype=float)
        chosen = rng.choice(n,size=min(n,int(p2)),replace=False)
        values[chosen] = 1
        initial_degrees = np.array([graph.degree(i) for i in range(n)],dtype=float)
        capacity = (1+p1)*np.maximum(initial_degrees,1)
    frames=[values.copy()]
    series=[float(values.mean()) if MODE=="opinion" else int(np.count_nonzero(values))]
    with tqdm(total=steps,desc="Simulation",unit="step") as bar:
        for _ in range(steps):
            new=values.copy()
            if MODE=="diffusion":
                for i in range(n):
                    if values[i]==1:
                        if rng.random()<p2: new[i]=0
                        for j in neighbors[i]:
                            if values[j]==0 and rng.random()<p1: new[j]=1
            elif MODE=="opinion":
                for i in range(n):
                    if not len(neighbors[i]): continue
                    avg=float(values[neighbors[i]].mean())
                    diff=avg-values[i]
                    # Agree when nearby, push away when far apart.
                    new[i]=np.clip(values[i]+p1*diff*(1 if abs(diff)<p2 else -.4),-1,1)
            else:
                active = values==0
                for i in range(n):
                    if not active[i]: continue
                    # Failed neighbors transfer one unit of demand each.
                    load = initial_degrees[i] + sum(values[j]==1 for j in neighbors[i])
                    if load>capacity[i]: new[i]=1
            values=new
            frames.append(values.copy())
            series.append(float(values.mean()) if MODE=="opinion" else int(np.count_nonzero(values)))
            bar.update(1)
            if MODE=="failure" and np.array_equal(frames[-1],frames[-2]): break
    return graph,positions,frames,series

def save_results(graph,positions,frames,series,settings):
    with (OUT/"settings.json").open("w") as f: json.dump(settings,f,indent=2)
    with (OUT/"metrics.csv").open("w",newline="") as f:
        writer=csv.writer(f); writer.writerow(["step","metric"])
        writer.writerows(enumerate(series))
    fig,ax=plt.subplots(figsize=(6,3.2))
    ax.plot(range(len(series)),series,color="#cf4949")
    ax.set(xlabel="Step",ylabel="Mean opinion" if MODE=="opinion" else "Affected nodes",title="Cascading Failures")
    fig.tight_layout();fig.savefig(OUT/"metrics.png",dpi=140);plt.close(fig)
    fig,ax=plt.subplots(figsize=(6,6),facecolor="#10141d")
    ax.set_facecolor("#10141d")
    xy=np.array([positions[i] for i in range(len(graph))])
    edges=np.array(list(graph.edges()),dtype=int)
    if len(edges):
        from matplotlib.collections import LineCollection
        lines=LineCollection(xy[edges],colors="#536173",linewidths=.5,alpha=.45)
        ax.add_collection(lines)
    scatter=ax.scatter(xy[:,0],xy[:,1],s=max(9,900/len(graph)),linewidths=0,zorder=2)
    ax.set_xlim(xy[:,0].min()-.15,xy[:,0].max()+.15)
    ax.set_ylim(xy[:,1].min()-.15,xy[:,1].max()+.15)
    ax.set_aspect("equal");ax.axis("off")
    label=ax.text(.02,1.01,"",transform=ax.transAxes,color="white")
    stride=max(1,math.ceil(len(frames)/160))
    indices=list(range(0,len(frames),stride))
    if indices[-1]!=len(frames)-1:indices.append(len(frames)-1)
    def draw(k):
        values=frames[k]
        if MODE=="opinion":
            scatter.set_array(values)
            scatter.set_cmap("coolwarm")
            scatter.set_clim(-1,1)
            label.set_text(f"Step {k}/{settings['steps']} | Mean {values.mean():+.2f}")
        else:
            scatter.set_array(values)
            scatter.set_cmap("Reds")
            scatter.set_clim(0,1)
            label.set_text(f"Step {k}/{settings['steps']} | Affected {int(values.sum())}/{len(values)}")
        return scatter,label
    ani=FuncAnimation(fig,draw,frames=indices,interval=1000/15,blit=False,repeat=False)
    with tqdm(total=len(indices),desc="Rendering GIF",unit="frame") as progress:
        class Writer(PillowWriter):
            def grab_frame(self,**kwargs):
                super().grab_frame(**kwargs);progress.update(1)
        ani.save(str(OUT/"simulation.gif"),writer=Writer(fps=15),dpi=90)
    plt.close(fig)
    print("Saved:",OUT/"simulation.gif")
    display(Image(filename=str(OUT/"simulation.gif")))

def upload_outputs():
    from google.colab import userdata
    import requests,base64
    token=userdata.get("GITHUB_TOKEN")
    if not token: raise ValueError("Colab SecretsにGITHUB_TOKENを設定してください")
    headers={"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json"}
    for file in sorted(OUT.iterdir()):
        if file.suffix not in (".gif",".png",".csv",".json"):continue
        remote=f"{PREFIX}/outputs/{file.name}"
        url=f"https://api.github.com/repos/{REPO}/contents/{remote}"
        existing=requests.get(url,headers=headers,timeout=30)
        if existing.status_code not in (200,404):existing.raise_for_status()
        payload={"message":f"Update Cascading Failures output","content":base64.b64encode(file.read_bytes()).decode()}
        if existing.status_code==200:payload["sha"]=existing.json()["sha"]
        response=requests.put(url,headers=headers,json=payload,timeout=120);response.raise_for_status()
        print("Uploaded:",remote)


## Controls

設定を変更して実行すると、進捗・GIF・CSV・グラフが生成されます。GitHubアップロードにはColab Secretsの `GITHUB_TOKEN` が必要です。

In [ ]:
style={"description_width":"115px"}
n=widgets.IntSlider(value=120,min=20,max=500,step=10,description="ノード数",style=style,continuous_update=False)
steps=widgets.IntSlider(value=100,min=10,max=400,step=10,description="ステップ数",style=style,continuous_update=False)
degree=widgets.IntSlider(value=4,min=1,max=12,description="接続数の目安",style=style,continuous_update=False)
graph_type=widgets.Dropdown(options=["Scale-free","Small-world","Random"],description="ネットワーク",style=style)
p1=widgets.FloatSlider(value=0.25,min=0.05,max=1,step=0.05,description="負荷余裕",style=style,continuous_update=False)
p2=widgets.IntSlider(value=3,min=1,max=20,step=1,description="初期故障数",style=style,continuous_update=False)
seed=widgets.IntText(value=42,description="乱数シード",style=style)
upload=widgets.Checkbox(value=False,description="GitHubに結果をアップロード")
button=widgets.Button(description="シミュレーション実行・GIF生成",button_style="primary")
output=widgets.Output()
def start(_):
    button.disabled=True
    with output:
        clear_output(wait=True)
        try:
            settings=dict(nodes=n.value,steps=steps.value,degree=degree.value,graph=graph_type.value,
                          parameter1=p1.value,parameter2=p2.value,seed=seed.value)
            graph,positions,frames,series=run_model(n.value,steps.value,seed.value,graph_type.value,
                                                     degree.value,p1.value,p2.value)
            print(f"計算終了: {len(frames)-1} steps / {len(graph)} nodes")
            save_results(graph,positions,frames,series,settings)
            if upload.value:upload_outputs()
            else:print("READMEへのGIF反映にはGitHubアップロードを有効にしてください。")
        except Exception as e:
            print(f"Error: {type(e).__name__}: {e}")
            raise
        finally:button.disabled=False
button.on_click(start)
display(widgets.VBox([widgets.HTML("<h3>Cascading Failures</h3>"),graph_type,n,degree,steps,p1,p2,seed,upload,button,output]))
